# Project: Yellow Taxi Trip - ETL Pipeline

### Dataset: Yellow Taxi Trip Records - Março 2016

In [1]:
#Realizando as importações das bibliotecas

import pandas as pd
import numpy as np
from pathlib import Path

#### 1. Extract - Leitura dos dados



In [2]:
#Iniciando DataFrame

PATH_INPUT = Path.cwd().parent / 'data'

PATH_FILE = PATH_INPUT.iterdir()

for arquivo in PATH_FILE:
    if arquivo.is_file():
        PATH_FILE = arquivo



df = pd.read_csv(PATH_FILE)

In [3]:
# Analisando estrutura do Df

print(f"Número de linhas: {df.shape[0]} \nNúmero de colunas: {df.shape[1]}\n")


print("Vendo o HEAD do df:")

df.head()


Número de linhas: 12210952 
Número de colunas: 19

Vendo o HEAD do df:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01 00:00:00,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,2016-03-01 00:00:00,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2016-03-01 00:00:00,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,2,2016-03-01 00:00:00,2016-03-01 00:00:00,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,2,2016-03-01 00:00:00,2016-03-01 00:00:00,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,98.0,0.0,0.0,0.00,15.50,0.3,113.80


In [1]:
# Analisando a estrutura do df

print("Tipos dos dados:")

df.info()


# Coerção de datas

df['pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

Tipos dos dados:


NameError: name 'df' is not defined

#### 2. Tratamentos dos dados

In [5]:
# Identificação de valores nulos:

null_values = df.isnull().sum()

print(null_values[null_values > 0] if null_values.sum() > 0 else "Não há valores nulos")

Não há valores nulos


In [6]:
# Identificação de anomalias


# número de passageiros invalido

passageiros_invalidos = (df['passenger_count'] <= 0) | (df['passenger_count'] > 6)


# distancias maiores que o normal (ou menores que o normal)

distancias_invalidas = (df['trip_distance'] <= 0) | (df['trip_distance'] > 100)


# hora de saida menor que a hora de entrada

horas_invalidas = df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']



#tarifa negativa

tarifa_negativa = df['fare_amount'] < 0


# corridas fora de NYC

coords_entrada_invalidas = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
) 

coords_saida_invalidas =  (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.3)
)


coord = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
) | (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.3)
)






# RESULTADO DAS ANOMALIAS:

print(f"Quantidade de passageiros invalidos: {passageiros_invalidos.sum()} ({passageiros_invalidos.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de distancias invalidas: {distancias_invalidas.sum()} ({distancias_invalidas.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de horas invalidas: {horas_invalidas.sum()} ({horas_invalidas.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de tarifas invalidas: {tarifa_negativa.sum()} ({tarifa_negativa.sum()/df.shape[0] *100: .5f} %)")
print(f"Quantidade de pickup fora de NYC: {coords_entrada_invalidas.sum()} ({coords_entrada_invalidas.sum()/df.shape[0] *100: .5f} %)")
print(f"Quantidade de dropoff fora de NYC: {coords_saida_invalidas.sum()} ({coords_saida_invalidas.sum()/df.shape[0] *100: .5f} %)")



Quantidade de passageiros invalidos: 678 ( 0.00555 %)
Quantidade de distancias invalidas: 71225 ( 0.58329 %)
Quantidade de horas invalidas: 12550 ( 0.10278 %)
Quantidade de tarifas invalidas: 4581 ( 0.03752 %)
Quantidade de pickup fora de NYC: 183816 ( 1.50534 %)
Quantidade de dropoff fora de NYC: 174501 ( 1.42905 %)


In [ ]:
#Retirando os registros invalidos do DF

filtro_invalidos = (
    passageiros_invalidos |
    distancias_invalidas |
    horas_invalidas |
    tarifa_negativa |
    coords_entrada_invalidas |
    coords_saida_invalidas 
)

df_filtrado = df[~filtro_invalidos].copy()

linhas_removidas = df.shape[0] - df_filtrado.shape[0]

print(f"Quantidade de linhas antes: {df.shape[0]}")


print(f"Quantidade de linhas removidas: {linhas_removidas} ({linhas_removidas/df.shape[0] *100: .2f} %)")

print(f"Quantidade de linhas depois: {df_filtrado.shape[0]}")

#### 3. Enriquecimento dos dados

#### 4. Agregações

#### 5. Load